# Date Fruit Classification

Classify date fruit varieties from numerical measurements.

See [README.md](README.md) for data, setup, workflow, and evaluation notes.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("date_fruit.csv")

In [ ]:
df.head() #34 features,1 Category

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop("Class",axis=1)
y = df["Class"]

In [ ]:
df["Class"].unique()

In [ ]:
from sklearn.preprocessing import StandardScaler,LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
print("Original features:", X_train_scaled.shape[1])
print("PCA features:", X_train_pca.shape[1])

In [ ]:
#ANN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [ ]:
X_train_tensor = torch.tensor(X_train_pca,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.long)

X_test_tensor = torch.tensor(X_test_pca,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.long)

In [ ]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [ ]:
#Build our Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train_pca.shape[1],64),
            nn.ReLU(),

            nn.Linear(64,64),
            nn.ReLU(),

            nn.Linear(64,7)
        )

    def forward(self,x):
        return self.model(x)

In [ ]:
model = ANN()

#loss & optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
#Training the NN

epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb,yb in train_loader:

        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    training_loss = running_loss/len(train_loader)

    print(f"epoch = {epoch + 1}/{epochs} , loss = {training_loss}")

In [ ]:
#Evaluation

model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs = model(xb) #[0.2,0.3,0.6......]
        _, predicted = torch.max(outputs,1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print("Total Values : ",total)
print("Total Correct : ",correct)
print("Accuracy : ",correct/total * 100)